# LC 212 — Word Search II
**Difficulty:** Hard | **Category:** Tries / Prefix Trees
**Pattern:** Trie built from word list + DFS on grid with backtrack

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Build a Trie from all target words
first. Then DFS every grid cell — at each step, follow the Trie
node matching the current letter. If the node has no child for
the next letter, prune immediately. If is_end is True, collect
the word. One DFS pass finds ALL words simultaneously.
</div>

## Official Problem Statement

Given an `m x n` board of characters and a list of strings `words`,
return all words on the board.

Each word must be constructed from letters of sequentially adjacent
cells, where adjacent cells are horizontally or vertically
neighboring. The same letter cell may not be used more than once
in a word.

**Constraints:**
- `m == board.length`, `n == board[i].length`
- `1 <= m, n <= 12`
- `board[i][j]` is a lowercase English letter.
- `1 <= words.length <= 3 * 10^4`
- `1 <= words[i].length <= 10`
- `words[i]` consists of lowercase English letters.
- All the strings in `words` are unique.

## What This Is Actually Asking

Find every word from a list that can be spelled by walking
adjacently through a letter grid (no revisiting a cell per word).

A naive approach would run Word Search I (LC 79) for each word
separately — way too slow with 30,000 words.

The key is to build a Trie from all words first, then do one DFS
per starting cell. At each cell, follow the Trie node for the
current letter. If the Trie has no path forward, stop exploring
that direction immediately — this prunes dead-end branches for
ALL words at once, not just one.

## Walk Through an Example by Hand

```
Board:          words = ["oath", "eat", "rain"]
  o  a  a  n
  e  t  a  e
  i  h  k  r
  i  f  l  v

Step 1: Build Trie from ["oath", "eat", "rain"]
  root -> o -> a -> t -> h [end='oath']
       -> e -> a -> t     [end='eat']
       -> r -> a -> i -> n [end='rain']

Step 2: DFS from cell (0,0) = 'o'
  Trie root has 'o'? Yes. Follow it.
  Mark (0,0) as visited ('#')
  Neighbours of (0,0): (0,1)='a', (1,0)='e'
    -> (0,1)='a': Trie 'o' node has 'a'? Yes.
      Mark (0,1). Neighbours: (0,2)='a', (1,1)='t'
        -> (1,1)='t': Trie 'oa' node has 't'? Yes.
          Mark (1,1). Neighbour (2,1)='h'.
            -> (2,1)='h': is_end=True! Collect "oath".
  Unmark cells as we backtrack.

Result: ["oath", "eat"]  ("rain" found from cell (0,3))
```

## The Picture

Trie from words ["oath", "eat", "rain"]:
```
root
 ├── 'o'
 │    └── 'a'
 │         └── 't'
 │              └── 'h' [is_end, word="oath"]
 ├── 'e'
 │    └── 'a'
 │         └── 't' [is_end, word="eat"]
 └── 'r'
      └── 'a'
           └── 'i'
                └── 'n' [is_end, word="rain"]
```

DFS visits a cell, marks it '#' to block revisit:
```
board[r][c] = '#'     <- mark visited
... recurse neighbours ...
board[r][c] = letter  <- restore (backtrack)
```

Trie node also stores the word string at is_end nodes
so we don't need to reconstruct it during DFS:
```
node.word = "oath"  # set during insert, not None
```

## When To Use This Pattern

- When you see **find multiple words in a grid**, think Trie + DFS.
- When running Word Search I per word would be too slow, think Trie
  to **search all words simultaneously**.
- When you need **early pruning** of grid paths, think Trie child
  lookup — no child means stop immediately.
- When the **word list is large** and words share prefixes, a Trie
  amortizes the shared prefix cost.
- When a **backtracking DFS** needs a guide to prune, think Trie
  node as the navigator.

## The Approach

Build a Trie from all words. Store the full word string at end
nodes so we can collect it directly without reconstruction.

Then iterate every cell of the board as a potential start. Launch
a DFS from cells whose letter exists in Trie's root children.

In DFS: mark cell visited with '#', recurse all four neighbours
that exist in the current Trie node's children, restore the cell
on backtrack. If current node has a word, add to results and clear
it from the Trie (to avoid duplicates and prune future DFS).

Optionally prune Trie nodes with no children after collecting a
word — this significantly speeds up subsequent DFS calls.

In [ ]:
# Standard library only
from typing import List   # for type hints


In [ ]:
def test_harness(func):
    """
    Each test: (board, words, expected_list).
    Comparison uses sorted() to ignore order.
    """
    tests = [
        # --- Test 1: LeetCode example 1 ---
        (
            [
                ["o","a","a","n"],
                ["e","t","a","e"],
                ["i","h","k","r"],
                ["i","f","l","v"]
            ],
            ["oath","pea","eat","rain"],
            ["eat", "oath"]
        ),
        # --- Test 2: LeetCode example 2 ---
        (
            [["a","b"],["c","d"]],
            ["abcd"],
            []
        ),
        # --- Test 3: single cell board ---
        (
            [["a"]],
            ["a", "b"],
            ["a"]
        ),
        # --- Test 4: word not on board ---
        (
            [["a","b"],["c","d"]],
            ["abdc"],
            ["abdc"]
        ),
        # --- Test 5: multiple words, shared prefix in Trie ---
        (
            [
                ["o","a","n"],
                ["t","d","e"],
                ["s","x","k"]
            ],
            ["oat", "oats", "note"],
            ["oat", "oats"]
        ),
    ]

    passed = 0
    failed = 0

    for i, (board, words, expected) in enumerate(tests, 1):
        result = func([row[:] for row in board], words)
        if sorted(result) == sorted(expected):
            print(f"  PASSED Test {i}")
            passed += 1
        else:
            print(
                f"  FAILED Test {i}: got {sorted(result)}, "
                f"expected {sorted(expected)}"
            )
            failed += 1

    print(f"\nResults: {passed} passed, {failed} failed "
          f"out of {passed + failed} tests")


In [ ]:
class TrieNode:
    """Trie node: children dict + optional stored word."""
    def __init__(self):
        self.children = {}   # char -> TrieNode
        self.word     = None # str if word ends here, else None


def findWords(board: List[List[str]],
              words: List[str]) -> List[str]:
    """
    Find all words from the list that exist in the board.

    Strategy:
        1. Build Trie from all words (store word at end node).
        2. DFS from every cell, guided by Trie nodes.
        3. Mark cells visited with '#'; restore on backtrack.
        4. When node.word is set, collect it and clear to avoid
           duplicates; optionally prune empty Trie nodes.

    Time:  O(m * n * 4 * 3^(L-1)) where L = max word length
    Space: O(W * L) for Trie, W = number of words
    """
    print(f"[DEBUG] board size: {len(board)}x{len(board[0])}")
    print(f"[DEBUG] word count: {len(words)}")
    print(f"[DEBUG] words: {words}")

    # Step 1: Build Trie
    # TODO: implement
    root = TrieNode()
    print("[DEBUG] Trie root created — insert words here")

    # Step 2: DFS helper
    result = []

    def dfs(node, r, c):
        print(f"  [DEBUG] dfs r={r} c={c} char={board[r][c]}")
        pass  # TODO: implement

    # Step 3: Launch DFS from every cell
    rows, cols = len(board), len(board[0])
    for r in range(rows):
        for c in range(cols):
            pass  # TODO: call dfs if board[r][c] in root.children

    return result


In [ ]:
# Uncomment and run when solution is ready
# test_harness(findWords)


## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute (Word Search I per word) | O(W * m*n * 4 * 3^L) | O(L) |
| **Trie + DFS (optimal)** | **O(m*n * 4 * 3^L)** | **O(W*L)** |

- m, n = board dimensions
- L = maximum word length
- W = number of words
- The Trie eliminates the W multiplier — all words explored in
  one DFS pass. Trie pruning further reduces real-world cost.
- 4 * 3^(L-1): 4 directions from start, then 3 (can't go back)

## Real World Connection

At Citi, detecting anomalous API call sequences across 6,000
endpoints is structurally similar: build a Trie of known bad
patterns, then scan incoming event streams for matches. The Trie
prunes irrelevant paths immediately — exactly like grid DFS pruning
on Trie miss.

An ETL pipeline ingesting AWS CloudWatch logs can use a Trie-guided
scan to extract structured fields from unstructured log lines. Each
log character is a "grid cell"; known field patterns are Trie words.
One pass finds all matching patterns simultaneously.

Prophet ML forecasting stores results across a matrix of
(endpoint, time_bucket) pairs — conceptually a 2D grid. Finding
which forecasts match a set of naming patterns (anomaly labels) is
exactly the Word Search II problem: Trie from patterns, DFS over
the (endpoint x time) space with early pruning.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra